In [2]:
import numpy as np
import torch

# -------------------------
# 1D GRF sampler (periodic)
# -------------------------
def sample_grf_1d(Nx, L=1.0, alpha=2.0, tau=3.0, sigma=0.218, rng=None):
    """
    Periodic 1D Gaussian random field on Nx points over [0, L).
    Power spectrum ~ (tau^2 + (2πk)^2)^(-alpha/2).

    NOTE: sigma=0.218 is chosen to be closer to your 16-res amplitude scale.
          (Your 100-res code used sigma=1.0 -> much larger amplitudes.)
    """
    if rng is None:
        rng = np.random.default_rng()

    k = np.fft.fftfreq(Nx, d=L / Nx)  # cycles per unit length
    lam = (tau**2 + (2.0 * np.pi * k) ** 2) ** (-alpha / 2.0)

    z = rng.normal(size=Nx) + 1j * rng.normal(size=Nx)
    u_hat = z * np.sqrt(lam)

    u = np.fft.ifft(u_hat).real
    u = (u - u.mean()) / (u.std() + 1e-12)
    u = sigma * u
    return u.astype(np.float64)


# ---------------------------------------------------
# Inviscid Burgers solver (FV + local LLF/Rusanov flux)
# u_t + (1/2 u^2)_x = 0
# ---------------------------------------------------
def solve_inviscid_burgers_fv_llf(u0, L=1.0, t_max=1.0, Nx=100, Nt_out=100, CFL=0.6):
    """
    Returns snapshots u(t_j, x_i) with shape (Nt_out, Nx), on [0,L) periodic grid.
    Uses internal stable dt from CFL, then records Nt_out evenly spaced snapshots.

    Scheme:
      u^{n+1}_i = u^n_i - (dt/dx) (F_{i+1/2} - F_{i-1/2})
      F_{i+1/2} = 0.5(f(u_i)+f(u_{i+1})) - 0.5*a_{i+1/2}(u_{i+1}-u_i)
      f(u)=0.5u^2,  a_{i+1/2}=max(|u_i|,|u_{i+1}|)
    """
    assert u0.shape == (Nx,)
    dx = L / Nx
    u = u0.copy()

    # choose dt from CFL based on max wave speed ~ max|u|
    max_u0 = max(np.max(np.abs(u)), 1e-8)
    dt = CFL * dx / max_u0

    n_steps = int(np.ceil(t_max / dt))
    dt = t_max / n_steps  # hit t_max exactly

    save_times = np.linspace(0.0, t_max, Nt_out)
    snapshots = np.empty((Nt_out, Nx), dtype=np.float64)

    t = 0.0
    snapshots[0] = u.copy()
    save_idx = 1

    for step in range(n_steps):
        u_prev = u

        # periodic neighbors
        uR = np.roll(u_prev, -1)
        # flux f(u)=0.5 u^2
        f = 0.5 * u_prev**2
        fR = np.roll(f, -1)

        # local wave speed
        a = np.maximum(np.abs(u_prev), np.abs(uR))

        # LLF fluxes at i+1/2
        F_iphalf = 0.5 * (f + fR) - 0.5 * a * (uR - u_prev)
        F_imhalf = np.roll(F_iphalf, 1)

        # update
        u = u_prev - (dt / dx) * (F_iphalf - F_imhalf)
        t += dt

        if not np.all(np.isfinite(u)):
            raise RuntimeError(f"Non-finite at step {step}, t={t:.4e}. Lower CFL or reduce IC amplitude.")

        # save snapshots when passing save_times
        while save_idx < Nt_out and t + 1e-15 >= save_times[save_idx]:
            snapshots[save_idx] = u.copy()
            save_idx += 1
        if save_idx >= Nt_out:
            break

    while save_idx < Nt_out:
        snapshots[save_idx] = u.copy()
        save_idx += 1

    return snapshots


def build_and_save(split_name, N, Nx=100, Nt_out=100, L=1.0, t_max=1.0, seed=0,
                   sigma=0.218, out_path="burgers_train_100_inviscid.pt"):
    rng = np.random.default_rng(seed)

    X = np.empty((N, Nx), dtype=np.float64)          # ICs
    Y = np.empty((N, Nt_out, Nx), dtype=np.float64)  # trajectories

    for i in range(N):
        u0 = sample_grf_1d(Nx, L=L, alpha=2.0, tau=3.0, sigma=sigma, rng=rng)
        traj = solve_inviscid_burgers_fv_llf(u0, L=L, t_max=t_max, Nx=Nx, Nt_out=Nt_out, CFL=0.6)

        X[i] = u0
        Y[i] = traj

        if (i + 1) % 50 == 0:
            print(f"[{split_name}] {i+1}/{N} done")

    data = {
        "x": torch.from_numpy(X),               # [N, Nx]
        "y": torch.from_numpy(Y),               # [N, Nt_out, Nx]
        "visc": torch.zeros((N,), dtype=torch.float64),  # inviscid => 0, keep key for compatibility
    }
    torch.save(data, out_path)
    print(f"Saved {split_name} → {out_path}")
    print("Shapes:", data["x"].shape, data["y"].shape, data["visc"].shape)
    print("x std:", data["x"].std().item(), "y std:", data["y"].std().item())


if __name__ == "__main__":
    build_and_save("train", N=800, Nx=100, Nt_out=100, seed=123, sigma=0.218,
                   out_path="burgers_train_100_inviscid.pt")
    build_and_save("test", N=400, Nx=100, Nt_out=100, seed=999, sigma=0.218,
                   out_path="burgers_test_100_inviscid.pt")


[train] 50/800 done
[train] 100/800 done
[train] 150/800 done
[train] 200/800 done
[train] 250/800 done
[train] 300/800 done
[train] 350/800 done
[train] 400/800 done
[train] 450/800 done
[train] 500/800 done
[train] 550/800 done
[train] 600/800 done
[train] 650/800 done
[train] 700/800 done
[train] 750/800 done
[train] 800/800 done
Saved train → burgers_train_100_inviscid.pt
Shapes: torch.Size([800, 100]) torch.Size([800, 100, 100]) torch.Size([800])
x std: 0.21800136241980897 y std: 0.1762139642452116
[test] 50/400 done
[test] 100/400 done
[test] 150/400 done
[test] 200/400 done
[test] 250/400 done
[test] 300/400 done
[test] 350/400 done
[test] 400/400 done
Saved test → burgers_test_100_inviscid.pt
Shapes: torch.Size([400, 100]) torch.Size([400, 100, 100]) torch.Size([400])
x std: 0.21800272496139908 y std: 0.17628563744505207
